<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/fiscal/Divida-Liquida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dívida líquida do setor público — % do PIB

Este notebook é o motor de coleta e publicação da série `divida-liquida-pib`.

- **Fonte:** Banco Central do Brasil — Estatísticas Fiscais, via API de Dados Abertos do BCB (SGS), série **4513** — “Dívida líquida do setor público consolidado, % do PIB”.
- **Endpoint:** `https://api.bcb.gov.br/dados/serie/bcdata.sgs.4513/dados?formato=json`
- **Definição:** estoque da dívida líquida do setor público consolidado no fim do mês, descontados os ativos financeiros (inclusive reservas), em % do PIB acumulado em 12 meses. É uma razão, não um total: `eixo: percentual`, `medida: nivel`, unidade `percentual` (`%`, 1 casa), granularidade `mensal`.
- **Fórmula:** nenhuma. O cálculo é definido pela fonte; o notebook busca a série oficial, descarta o mês corrente (ainda aberto) e formata as datas no primeiro dia do mês (C15).
- **Período publicado:** de `2001-12-01` até o último mês fechado disponível.
- **Arquivo gerado:** `indicadores/fiscal/divida-liquida-pib.json`, com `versao: 1` e `id: divida-liquida-pib`.

## O que o número é — e o que não é

A dívida líquida é a dívida do setor público consolidado — União, INSS, estados, municípios, estatais (exceto Petrobras e Eletrobras) e Banco Central — descontados os ativos financeiros, inclusive as reservas internacionais. Não é a dívida bruta do governo geral (DBGG): além do desconto de ativos, o universo é outro (a DBGG cobre o governo geral, sem BCB e estatais). Meses recentes são preliminares e o histórico pode ser revisado quando o IBGE revisa o PIB — é revisão da fonte, não erro de fórmula. Não é dívida externa e não mede “qualidade de gestão”.

## Coleta da série oficial

A leitura é feita desde o início da série, sem atualização incremental e sem ler o JSON já publicado para decidir o período. O SGS devolve `data` em `DD/MM/AAAA` e `valor` como texto; o valor é convertido para `float` sem arredondar.

In [1]:
from datetime import date, datetime
from pathlib import Path
import json
import math
import os
import re
import time

import pandas as pd
import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()

if os.environ.get("DADOS_RAIZ"):
    PASTA = RAIZ / "indicadores" / "fiscal"
    if not PASTA.is_dir():
        raise SystemExit(f"família não encontrada em {PASTA}")
else:
    PASTA = RAIZ

DESTINO = Path(os.environ.get("FISCAL_SAIDA", PASTA)).resolve()
ARQUIVO = DESTINO / "divida-liquida-pib.json"


def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name

In [2]:
SERIE = 4513
URL = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{SERIE}/dados"
URL_FONTE = f"{URL}?formato=json"
HOJE = date.today()

for tentativa in range(4):
    try:
        resposta = requests.get(URL, params={"formato": "json"}, timeout=60)
        resposta.raise_for_status()
        bruto = resposta.json()
        if not isinstance(bruto, list) or not bruto:
            raise ValueError("resposta vazia ou fora do formato de lista")
        break
    except (requests.RequestException, ValueError):
        if tentativa == 3:
            raise
        time.sleep(2 ** tentativa)

df = pd.DataFrame(
    {
        "data": pd.to_datetime(
            [datetime.strptime(item["data"], "%d/%m/%Y") for item in bruto]
        ),
        "valor": [float(item["valor"]) for item in bruto],
    }
).sort_values("data", ignore_index=True)

print(f"Série SGS {SERIE}: {len(df)} leituras")
print(f"Período bruto: {df['data'].min():%Y-%m-%d} a {df['data'].max():%Y-%m-%d}")
print(f"Arquivo de saída: {relativo(ARQUIVO)}")
print("\nPrimeiras 5 leituras:")
print(df.head().to_string(index=False))
print("\nÚltimas 5 leituras:")
print(df.tail().to_string(index=False))

Série SGS 4513: 298 leituras
Período bruto: 2001-12-01 a 2026-09-01
Arquivo de saída: indicadores/fiscal/divida-liquida-pib.json

Primeiras 5 leituras:
      data  valor
2001-12-01  51.49
2002-01-01  52.28
2002-02-01  51.43
2002-03-01  51.01
2002-04-01  50.95

Últimas 5 leituras:
      data  valor
2026-05-01  67.88
2026-06-01  68.49
2026-07-01  69.11
2026-08-01  69.27
2026-09-01  69.98


## Filtragem dos meses fechados

O mês corrente é descartado porque ainda está aberto. A fonte devolve as datas no primeiro dia de cada mês; essa convenção é normalizada e preservada em C15.

In [3]:
primeiro_dia_mes_corrente = pd.Timestamp(HOJE.replace(day=1))
fechados = df[df["data"] < primeiro_dia_mes_corrente].copy()
fechados["mes"] = fechados["data"].dt.to_period("M")

if fechados.empty:
    raise ValueError("a fonte não retornou nenhum mês fechado")

if fechados["mes"].duplicated().any():
    raise ValueError("a fonte retornou meses repetidos")

print(f"Mês corrente excluído: {HOJE:%Y-%m}")
print(f"Meses fechados publicados: {len(fechados)}")
print(
    f"Primeiro: {fechados['data'].iloc[0]:%Y-%m-%d} "
    f"({fechados['valor'].iloc[0]:.2f}%)"
)
print(
    f"Último:   {fechados['data'].iloc[-1]:%Y-%m-%d} "
    f"({fechados['valor'].iloc[-1]:.2f}%)"
)

Mês corrente excluído: 2026-11
Meses fechados publicados: 298
Primeiro: 2001-12-01 (51.49%)
Último:   2026-09-01 (69.98%)


## Escrita do JSON canônico

Os pontos são gravados como `{"data": "YYYY-MM-DD", "valor": ...}`, em ordem ascendente, com a data no primeiro dia do mês e sem campos extras. O valor é o da fonte, sem arredondamento; `unidade.casas` é só exibição.

In [4]:
pontos = [
    {"data": f"{data.year:04d}-{data.month:02d}-01", "valor": float(valor)}
    for data, valor in zip(fechados["data"], fechados["valor"])
]

serie = {
    "versao": 1,
    "id": "divida-liquida-pib",
    "unidade": {"tipo": "percentual", "simbolo": "%", "casas": 1},
    "eixo": "percentual",
    "medida": "nivel",
    "granularidade": "mensal",
    "fonte": "Banco Central do Brasil — Estatísticas Fiscais (SGS 4513)",
    "urlFonte": URL_FONTE,
    "dados": pontos,
}

DESTINO.mkdir(parents=True, exist_ok=True)
with ARQUIVO.open("w", encoding="utf-8") as arquivo:
    json.dump(serie, arquivo, ensure_ascii=False, indent=2)

print(f"{len(pontos)} pontos escritos em {relativo(ARQUIVO)}")
print(f"Primeiro ponto: {pontos[0]}")
print(f"Último ponto:   {pontos[-1]}")

298 pontos escritos em indicadores/fiscal/divida-liquida-pib.json
Primeiro ponto: {'data': '2001-12-01', 'valor': 51.49}
Último ponto:   {'data': '2026-09-01', 'valor': 69.98}


## Validação inline

A célula final falha se o contrato, os metadados, a convenção C15, a unicidade dos meses, a ordem, a finitude dos valores, o primeiro ponto, os canários históricos (2014-05 e 2020-12) ou a igualdade com a fonte forem violados.

In [5]:
CHAVES = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}
assert set(serie) == CHAVES, sorted(set(serie) ^ CHAVES)
assert serie["versao"] == 1, serie["versao"]
assert serie["id"] == "divida-liquida-pib", serie["id"]
assert serie["unidade"] == {"tipo": "percentual", "simbolo": "%", "casas": 1}, serie["unidade"]
assert serie["eixo"] == "percentual", serie["eixo"]
assert serie["medida"] == "nivel", serie["medida"]
assert serie["granularidade"] == "mensal", serie["granularidade"]
assert serie["fonte"] == "Banco Central do Brasil — Estatísticas Fiscais (SGS 4513)", serie["fonte"]
assert serie["urlFonte"] == "https://api.bcb.gov.br/dados/serie/bcdata.sgs.4513/dados?formato=json", serie["urlFonte"]

datas = [ponto["data"] for ponto in serie["dados"]]
meses = [data[:7] for data in datas]

assert datas, "série vazia"
assert datas == sorted(datas), "datas fora de ordem"
assert all(a < b for a, b in zip(datas, datas[1:])), "datas não estritamente ascendentes"
assert len(meses) == len(set(meses)), "mês repetido"
assert datas[0] == "2001-12-01", datas[0]
assert serie["dados"][0]["valor"] == 51.49, serie["dados"][0]

# Canários independentes: pontos históricos fixos, fora do `fechados` que gerou a série.
por_data = {ponto["data"]: ponto["valor"] for ponto in serie["dados"]}
assert por_data["2014-05-01"] == 31.10, por_data["2014-05-01"]
assert por_data["2020-12-01"] == 61.37, por_data["2020-12-01"]

assert datas[-1][:7] < f"{HOJE:%Y-%m}", datas[-1]
assert len(datas) == len(fechados), (len(datas), len(fechados))

for ponto in serie["dados"]:
    assert set(ponto) == {"data", "valor"}, sorted(ponto)
    assert re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", ponto["data"]), ponto
    assert datetime.strptime(ponto["data"], "%Y-%m-%d").day == 1, ponto
    assert not isinstance(ponto["valor"], bool), ponto
    assert isinstance(ponto["valor"], (int, float)), ponto
    assert math.isfinite(ponto["valor"]), ponto

esperado_da_fonte = [
    {"data": f"{data.year:04d}-{data.month:02d}-01", "valor": float(valor)}
    for data, valor in zip(fechados["data"], fechados["valor"])
]
assert serie["dados"] == esperado_da_fonte, "valor publicado difere da fonte"

assert datas[-1] == esperado_da_fonte[-1]["data"], (datas[-1], esperado_da_fonte[-1])
assert serie["dados"][-1]["valor"] == esperado_da_fonte[-1]["valor"], serie["dados"][-1]

with ARQUIVO.open(encoding="utf-8") as arquivo:
    salvo = json.load(arquivo)
assert salvo == serie, "arquivo gravado difere do objeto validado"

print(
    f"ok: {len(datas)} meses únicos, {datas[0]} a {datas[-1]}, "
    f"valores idênticos à SGS 4513"
)

ok: 298 meses únicos, 2001-12-01 a 2026-09-01, valores idênticos à SGS 4513
